In [1]:
import flask_app


In [2]:
response = flask_app.load_data("../pickle_jar/api_data_20230607_173032.pkl")

In [4]:
df = pd.DataFrame(response.json())
df
API

,id,sport_key,sport_title,commence_time,home_team,away_team,bookmakers
0,e99a6a5ce1de542fe3899e92fc7362ff,baseball_mlb,MLB,2023-06-07T20:10:00Z,San Diego Padres,Seattle Mariners,"[{'key': 'draftkings', 'title': 'DraftKings', ..."
1,ba386e479057d0235b239c9e563f7c61,baseball_mlb,MLB,2023-06-07T22:10:00Z,Miami Marlins,Kansas City Royals,"[{'key': 'draftkings', 'title': 'DraftKings', ..."
2,e4a0d2c34db17061fe7b719232eed45a,baseball_mlb,MLB,2023-06-07T22:40:00Z,Tampa Bay Rays,Minnesota Twins,"[{'key': 'draftkings', 'title': 'DraftKings', ..."
3,d66c25c135e62675e003e7aafb45bd2a,baseball_mlb,MLB,2023-06-07T23:05:00Z,Washington Nationals,Arizona Diamondbacks,"[{'key': 'draftkings', 'title': 'DraftKings', ..."
4,7abd722003eaa564d5750dc3d89df8ec,baseball_mlb,MLB,2023-06-07T23:07:00Z,Toronto Blue Jays,Houston Astros,"[{'key': 'draftkings', 'title': 'DraftKings', ..."
5,037491356764f7d94fc56f84fcf26770,baseball_mlb,MLB,2023-06-07T23:10:00Z,Cleveland Guardians,Boston Red Sox,"[{'key': 'draftkings', 'title': 'DraftKings', ..."
6,481e40c6e25b400ba46c9101ac6b5ed9,baseball_mlb,MLB,2023-06-07T23:10:00Z,Cincinnati Reds,Los Angeles Dodgers,"[{'key': 'draftkings', 'title': 'DraftKings', ..."
7,e953d2e8b37318941316c33addb2941b,baseball_mlb,MLB,2023-06-07T23:20:00Z,Atlanta Braves,New York Mets,"[{'key': 'draftkings', 'title': 'DraftKings', ..."
8,f5eea25e889f4bd17532dd447e185bbc,baseball_mlb,MLB,2023-06-07T23:40:00Z,Milwaukee Brewers,Baltimore Orioles,"[{'key': 'draftkings', 'title': 'DraftKings', ..."
9,ff9f7e8292395b48689b49a07d4e6e74,baseball_mlb,MLB,2023-06-08T00:05:00Z,Texas Rangers,St. Louis Cardinals,"[{'key': 'draftkings', 'title': 'DraftKings', ..."


In [17]:
API_data = response

if type(API_data) == tuple:
    print("was tuple")
    headers = API_data[1]
    API_data = API_data[0]
else:
    print("was not tuple")
    headers = API_data.headers
    API_data = API_data.json()

    arbs = flask_app.find_arbs(API_data, qa_range=(2,25))
arbs

was not tuple


({}, 0, 0)

In [70]:
import glob
import os
import flask_app
import pickle
from datetime import datetime
from copy import deepcopy


In [2]:
def load_data(filename):
    data = None
    with open("../pickle_jar/"+filename, 'rb') as f:
        data = pickle.load(f)
    f.close()
    return data

In [90]:
# Convert the relative path to an absolute path
path_pattern = os.path.abspath('../pickle_jar/*.pkl')
print(f'Absolute path pattern: {path_pattern}')


curr = {}
open_arbs = {}
ded = []

# Use glob to get all filenames matching the pattern
file_list = glob.glob(path_pattern)

# Check if any files were found
if not file_list:
    print(f'No files found for pattern: {path_pattern}')
else:
    # Iterate over the filenames
    for filename in sorted(file_list):
        base_filename = os.path.basename(filename)
        print(f"..//pickle_jar//{base_filename}")
        API_data = load_data(base_filename)

        if type(API_data) == tuple:
            #print("was tuple")
            headers = API_data[1]
            API_data = API_data[0]
        else:
            #print("was not tuple")
            headers = API_data.headers
            API_data = API_data.json()


        print(headers["Date"])

        arbs = flask_app.find_arbs(API_data, qa_range=(2,25))
        curr = {}
        if arbs != ({}, 0, 0):
            #print(arbs)





            
            for sport in arbs[0]:
                #print(f"sport: {sport}")
                for event in arbs[0][sport]:
                    #print(f"event: {event}")#: {arb_i[0][sport][event]}")
                    for arb in arbs[0][sport][event]["arbs"]:
                        #print(arb)
                        key = f"{event}-{arb['market']}-{arb['line1']['book']}-{arb['line2']['book']}-{arb['line1']['price']}-{arb['line2']['price']}"
                        #print(key)

                        if key not in open_arbs:
                            # New arb!
                            curr[key] = {}
                            curr[key]["start_time"] = headers["Date"]
                            curr[key]["last_observation"] = headers["Date"]
                            curr[key]["details"] = arb
                            curr[key]["other"] = (sport, event)
                            print(f"new arb added: {key}")
                        else: # is in open already, existing arb rolls forward
                            # TODO update duration-so-far
                            curr[key] = open_arbs.pop(key)
                            curr[key]["last_observation"] = headers["Date"]
                            print(f"{key} already open! since { curr[key]['start_time']}. Now: {headers['Date']}")

                            


        # Done processing this API entry file
        # print(f"KILL QUEUE: {list(open_arbs.keys())}")

        # Whatever is left in open_arbs has died        
        for casualty in open_arbs:
            # casualty = dead arb key
            #TODO, handle data offload of arb closeout
            #TODO calculate time of death window
            open_arbs[casualty]["post_mortem"] = headers["Date"]
            #print(f"Dead arb: {casualty}")

            lifespan_min = f"{open_arbs[casualty]['last_observation']} - {open_arbs[casualty]['start_time']}"
            lifespan_max = f"{open_arbs[casualty]['post_mortem']} - {open_arbs[casualty]['start_time']}"


            #print(f"\t{lifespan_min}")
            #print(f"\t{lifespan_max}")
            
            date_format = '%a, %d %b %Y %H:%M:%S GMT'

            # Parse the date strings into datetime objects
            date2 = datetime.strptime(open_arbs[casualty]['last_observation'], date_format)
            date3 = datetime.strptime(open_arbs[casualty]['post_mortem'], date_format)
            date1 = datetime.strptime(open_arbs[casualty]['start_time'], date_format)

            lifespan_min = (date2 - date1).total_seconds() #/ 60
            lifespan_max = (date3 - date1).total_seconds() #/ 60


            if casualty in ded:
                print("ANOMOLY TO INSEPCT. SORTING OF FILE LIST ACCURATE?")

            ded.append(casualty)
        
            print(f"\t {casualty} Lifespan: ({lifespan_min},{lifespan_max})")

        open_arbs = deepcopy(curr)


    print(f"open: {len(open_arbs)}")
    print(f"ded: {len(ded)}")
    
        
    

Absolute path pattern: /home/R4RI/pickle_jar/*.pkl
..//pickle_jar//api_data_20240526_012841.pkl
Sun, 26 May 2024 01:28:41 GMT
..//pickle_jar//api_data_20240526_013214.pkl
Sun, 26 May 2024 01:32:14 GMT
new arb added: 7a7f4f2ce15ed061f8d5d2bd7395d79a-h2h-fanduel-espnbet-200--164
new arb added: 7a7f4f2ce15ed061f8d5d2bd7395d79a-h2h-betmgm-espnbet-230--164
..//pickle_jar//api_data_20240526_013222.pkl
Sun, 26 May 2024 01:32:22 GMT
7a7f4f2ce15ed061f8d5d2bd7395d79a-h2h-fanduel-espnbet-200--164 already open! since Sun, 26 May 2024 01:32:14 GMT. Now: Sun, 26 May 2024 01:32:22 GMT
7a7f4f2ce15ed061f8d5d2bd7395d79a-h2h-betmgm-espnbet-230--164 already open! since Sun, 26 May 2024 01:32:14 GMT. Now: Sun, 26 May 2024 01:32:22 GMT
..//pickle_jar//api_data_20240526_013239.pkl
Sun, 26 May 2024 01:32:39 GMT
7a7f4f2ce15ed061f8d5d2bd7395d79a-h2h-fanduel-espnbet-200--164 already open! since Sun, 26 May 2024 01:32:14 GMT. Now: Sun, 26 May 2024 01:32:39 GMT
new arb added: 7a7f4f2ce15ed061f8d5d2bd7395d79a-h2h-f

In [86]:
API_data = load_data("..//pickle_jar//api_data_20240526_013239.pkl")

if type(API_data) == tuple:
    #print("was tuple")
    headers = API_data[1]
    API_data = API_data[0]
else:
    #print("was not tuple")
    headers = API_data.headers
    API_data = API_data.json()

arbs = flask_app.find_arbs(API_data, qa_range=(2,25))
headers

{'Date': 'Sun, 26 May 2024 01:32:39 GMT', 'Content-Type': 'application/json; charset=utf-8', 'Content-Length': '429', 'Connection': 'keep-alive', 'X-Requests-Used': '108', 'X-Requests-Remaining': '392', 'X-Requests-Last': '1', 'vary': 'Accept-Encoding', 'content-encoding': 'gzip', 'Apigw-Requestid': 'YWuguifJIAMEP5Q='}